In [ ]:
import sys

sys.path.append("..")

from utils.orientation_map import *
from utils.generation import *
from utils.filters import *
from utils.density_map import *

import torch
import cv2

In [ ]:
def select_and_merge_density_maps(width, height):
    freq1 = load_random_density_map()
    freq2 = load_random_density_map()

    # Possibly load a third map
    flag_3 = rand() > 0.5
    freq3 = load_random_density_map() if flag_3 else None

    # Merge maps
    if flag_3:
        freq = (freq1 + freq2 + freq3) / 3 / 255.0
    else:
        freq = (freq1 + freq2) / 2 / 255.0

    # Resize to target dimensions, normalize to [0, 1]
    f_den = cv2.resize(freq, (width, height), interpolation=cv2.INTER_LINEAR)
    f_den = (f_den - np.min(f_den)) / (np.max(f_den) - np.min(f_den))

    return f_den

In [ ]:
width = 256
height = 256

def get_spiral_phase(psi, points, polarities):
    H, W = psi.shape
    Y, X = np.mgrid[:H, :W]
    zero = np.zeros_like(psi)
    for (yy, xx), p in zip(points, polarities):
        zero += p * np.arctan2(Y - yy, X - xx)

    return zero

def generate_heatmap(minutiae, h, w, sigma=3):
    """
    Converts list of (x, y) points into a Gaussian heatmap.
    """
    heatmap = torch.zeros((h, w))

    # Create a coordinate grid
    y_grid, x_grid = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")

    for x, y, _ in minutiae:
        # Simple Gaussian stamp
        # Check bounds
        if 0 <= x < w and 0 <= y < h:
            dist_sq = (x_grid - x) ** 2 + (y_grid - y) ** 2
            heatmap += torch.exp(-dist_sq / (2 * sigma**2))

    # Normalize heatmap to [0, 1] for stability
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    return heatmap.unsqueeze(0)  # Add channel dim -> (1, H, W)

In [ ]:
from model import FingerprintUNet

model = FingerprintUNet(in_channels=4, out_channels=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = torch.load('../checkpoints_exp/ckpt_full.pth', map_location=torch.device(device))

model.load_state_dict(checkpoint["model"])
model.to(device)
model.eval()

In [ ]:
singularity_type = 1


core_positions, delta_positions, arch_fact1, arch_fact2, k_arch = init_para_canonical(
    H=height, W=width, singularity_type=singularity_type
)
g_cap = set_param_canonical(singularity_type)

o_map = OrientationMap(
    width,
    height,
    singularity_type,
    delta_positions,
    core_positions,
    g_cap,
    arch_fact1,
    arch_fact2,
    k_arch,
)

orientation_map = o_map.getOrientationMap()

minutiae = []
target_count = 25
max_attempts = 1000
min_dist_sq = 16 ** 2  # 256. Comparing squared distances is faster!

attempts = 0
while len(minutiae) < target_count and attempts < max_attempts:
    attempts += 1
    y = random.randint(12, height - 12)
    x = random.randint(12, width - 12)
    
    # Assume the point is valid until proven otherwise
    is_valid = True
    
    for mx, my, _ in minutiae:
        # Calculate the squared Euclidean distance
        dist_sq = (x - mx)**2 + (y - my)**2
        
        if dist_sq < min_dist_sq:
            is_valid = False
            break  # It's too close to an existing point, abandon this (x, y)
            
    # If the point survived the checks against all existing minutiae, keep it
    if is_valid:
        minutiae.append((x, y, random.choice([-1, 1])))

cos_2_theta = torch.from_numpy(np.cos(2 * orientation_map)).float()
sin_2_theta = torch.from_numpy(np.sin(2 * orientation_map)).float()
minutiae_heatmap = generate_heatmap(minutiae, height, width, sigma=3)


freq_map = select_and_merge_density_maps(width, height)
freq_map_jitter = (
    (freq_map - np.min(freq_map)) / (np.max(freq_map) - np.min(freq_map)) * 0.05
)
base_freq = np.random.uniform(0.087, 0.133)
freq_map = base_freq + freq_map_jitter

In [ ]:
from model import FingerprintUNet
from loss_functions import FingerprintLossv2
from torch.utils.data import DataLoader, Subset
from dataset import FingerprintOrientationDataset


base_dir = "/data/hot/khangphuanhle/data_v3"

orientation_dir = "orientation_maps"
minutiae_dir = "minutiae_locations"
frequency_dir = "freq_maps"
cos_cont_dir = "cos_cont"
cos_full_dir = "cos_full"
sin_cont_dir = "sin_cont"
sin_full_dir = "sin_full"

orientation_paths = []
minutiae_paths = []
frequency_paths = []
cos_cont_paths = []
cos_full_paths = []
sin_cont_paths = []
sin_full_paths = []

for item in os.listdir(os.path.join(base_dir, orientation_dir)):
    if item.endswith(".npy"):
        cos_cont_paths.append(os.path.join(base_dir, cos_cont_dir, item))
        cos_full_paths.append(os.path.join(base_dir, cos_full_dir, item))
        sin_cont_paths.append(os.path.join(base_dir, sin_cont_dir, item))
        sin_full_paths.append(os.path.join(base_dir, sin_full_dir, item))
        minutiae_paths.append(
            os.path.join(base_dir, minutiae_dir, item.replace(".npy", ".txt"))
        )
        orientation_paths.append(os.path.join(base_dir, orientation_dir, item))
        frequency_paths.append(os.path.join(base_dir, frequency_dir, item))

assert (
    len(orientation_paths)
    == len(minutiae_paths)
    == len(frequency_paths)
    == len(cos_cont_paths)
    == len(cos_full_paths)
    == len(sin_cont_paths)
    == len(sin_full_paths)
), "Mismatch in dataset lengths."

orientation_paths.sort()
minutiae_paths.sort()
frequency_paths.sort()
cos_cont_paths.sort()
cos_full_paths.sort()
sin_cont_paths.sort()
sin_full_paths.sort()

train_split = int(0.85 * len(orientation_paths))

val_dataset = FingerprintOrientationDataset(
    orientation_paths[train_split:],
    minutiae_paths[train_split:],
    frequency_paths[train_split:],
    cos_cont_paths[train_split:],
    cos_full_paths[train_split:],
    sin_cont_paths[train_split:],
    sin_full_paths[train_split:],
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=True,
)

i = 3
with torch.no_grad():
    for inp in val_loader:
        if i > 0: 
            i -= 1
            continue

        inputs = inp["inputs"].to(device)  # (B, 3, H, W)

        pred = model(inputs)
        pred_sin = pred[:, 0:1, :, :].squeeze().cpu().numpy()
        pred_cos = pred[:, 1:2, :, :].squeeze().cpu().numpy()

        plt.figure(figsize=(14, 6))
        plt.subplot(1, 3, 1)
        plt.imshow(cos_2_theta.numpy(), cmap="gray")
        plt.title("Input Orientation Map Cosine")

        plt.subplot(1, 3, 2)
        plt.imshow(sin_2_theta.numpy(), cmap="gray")
        plt.title("Input Orientation Map Sine")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_cos, cmap="gray")
        plt.title("Predicted Cos Fingerpritn")
        plt.axis("off")
        plt.show()
       
        break

In [ ]:
sp = get_spiral_phase(
    orientation_map, [(y, x) for x, y, _ in minutiae], [p for _, _, p in minutiae]
)

with torch.no_grad():
    input_tensor = inputs.unsqueeze(0).to(device)  # Add batch dim
    output = model(input_tensor)
    pred_sin = output[:, 0:1, :, :].squeeze().cpu().numpy()
    pred_cos = output[:, 1:2, :, :].squeeze().cpu().numpy()

plt.figure(figsize=(14, 6))
plt.subplot(1, 3, 1)
plt.imshow(cos_2_theta.numpy(), cmap="gray")
plt.title("Input Orientation Map Cosine")

plt.subplot(1, 3, 2)
plt.imshow(sin_2_theta.numpy(), cmap="gray")
plt.title("Input Orientation Map Sine")

plt.subplot(1, 3, 3)
plt.imshow(pred_cos, cmap="gray")
plt.title("Predicted Cos Fingerpritn")
plt.axis("off")
plt.show()